# LLM-vetted multi-anchor theme entries

A step-by-step walkthrough of how a themed puzzle gets its theme words, and how the same theme still yields a *different* puzzle every time.

1. **Candidate pool** - score every database word against the theme (cosine) and take the top `candidate_pool` (all real crossword answers).
2. **LLM vetting** - one LLM call returns a large pool of genuinely on-theme words (up to `vetted_pool`), guarded by a two-tier validation.
3. **Random anchor draw** - the vetted pool is **unranked**: every word in it counts as equally on-theme, so the generator *samples* `max_anchors` of them at random rather than always taking the "best" few.
4. **Place and fill** - pin the drawn anchors at agreeing intersections and fill around them, redrawing when a draw cannot fill.

Steps 2 and 3 are what separate *relevance* from *ranking*: the LLM decides what is on-theme, and randomness decides which on-theme words this particular puzzle uses.

This is a code walkthrough: it peeks at the internal methods so the backend algorithm and the LLM call are visible under the hood.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, logging, random
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv; load_dotenv() # OPENAI_API_KEY: theme embedding + anchor-vetting call

logging.disable(logging.INFO) # keep the walkthrough output clean (hide INFO / HTTP logs)

In [ ]:
from src.gridgpt.word_database_manager import WordDatabaseManager
from src.gridgpt.template_manager import select_template
from src.gridgpt.theme_manager import ThemeManager
from src.gridgpt.theme_anchor import ThemeAnchorSelector
from src.gridgpt.crossword_generator import CrosswordGenerator, generate_themed_crossword
from src.gridgpt.utils import load_parameters

word_db = WordDatabaseManager()
cfg = load_parameters()['theme_anchors']
theme = 'food'
template = select_template(template_id='5x5_diagonal_cut')
cfg

## 1. The candidate pool (cosine shortlist)

`ThemeManager.get_anchor_candidates` scores every database word against the theme and returns the top `candidate_pool` words across all usable slot lengths (3&ndash;5). Every one is a real, published crossword answer &mdash; the raw material the LLM will vet.

In [ ]:
theme_manager = ThemeManager(theme, word_db)
candidates = theme_manager.get_anchor_candidates(
    pool_size=cfg['candidate_pool'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'],
)
print(f'{len(candidates)} cosine candidates for theme {theme!r}:')
candidates

## 2. The LLM vetting call (what the LLM is for)

Cosine is noisy: it ranks some loosely-related words highly (for "food" it surfaces FUN). One LLM call keeps only the words a solver would truly recognise as on-theme.

The important part is that it returns a **pool** (up to `vetted_pool`), not a ranked top-3. A big pool is what makes varied puzzles possible: `max_anchors` decides how many go into *one* grid, `vetted_pool` decides how many the grid can *choose from*.

In [ ]:
selector = ThemeAnchorSelector()
print('LLM connected:', selector.llm_connection_success)

pool = selector.select_anchors(
    theme, candidates, word_db,
    max_words=cfg['vetted_pool'], allow_llm_words=cfg['allow_llm_words'],
    min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'],
)
print(f'{len(candidates)} candidates -> {len(pool)} vetted on-theme words (all equally valid):')
pool

### Under the hood: the actual LLM call

`select_anchors` delegates the model call to the internal `_request_anchor_words`. Here is exactly what gets sent (system + user prompt, built from `conf/base/prompts.yml`) and the raw list the model returns, before any validation.

In [ ]:
# Rebuild the user prompt exactly as _request_anchor_words does, to see what the model reads.
own = selector.prompt['own_words_instructions_llm_not_allowed']
user_prompt = selector.prompt['user_prompt'].format(
    theme=theme,
    candidates='\n'.join(f'- {c}' for c in candidates),
    max_words=cfg['vetted_pool'],
    own_words_instruction=own,
)
print('--- SYSTEM PROMPT ---')
print(selector.prompt['system_prompt'])
print('--- USER PROMPT ---')
print(user_prompt)

# The raw model output, before the two-tier validation.
raw = selector._request_anchor_words(theme, candidates, cfg['vetted_pool'], allow_llm_words=False)
print('--- RAW LLM WORDS ---')
print(raw)

## 3. Two-tier validation (the guardrails)

Whatever the model returns is validated before it can become a grid entry, because the LLM can drift off-list or hallucinate:

- **Tier 1 (both modes):** a word in the database passes immediately. This drops hallucinations even when own-words is off.
- **Tier 2 (own-words only):** a word *not* in the database passes only when `allow_llm_words` is on and it is letters-only, in the 3&ndash;5 length range, and common enough (`wordfreq` Zipf &ge; `min_zipf`).

In [ ]:
from wordfreq import zipf_frequency

checks = ['PIZZA', 'MADEUPWORD', 'TACO', 'ZZZQX']
pd.DataFrame([{
    'word': w,
    'in DB (Tier 1)': ThemeAnchorSelector._in_database(w, word_db.word_list_with_frequencies),
    'zipf': round(zipf_frequency(w, 'en'), 2),
    'valid own-word (Tier 2)': ThemeAnchorSelector._is_valid_own_word(
        w, cfg['min_zipf'], cfg['min_chars'], cfg['max_chars']),
} for w in checks])

### Switching own-word suggestions on vs off

With `allow_llm_words=False` (default) only database words survive. With it on, the model may also add real, common words that are not in the database (their clue is then LLM-generated).

In [ ]:
def vet(allow):
    return selector.select_anchors(
        theme, candidates, word_db, max_words=cfg['vetted_pool'], allow_llm_words=allow,
        min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])

off, on = vet(False), vet(True)
print(f'allow_llm_words=False -> {len(off)} words: {off}')
print(f'allow_llm_words=True  -> {len(on)} words: {on}')
print('added by the LLM (not in the cosine candidates):', sorted(set(on) - set(candidates)))

## 4. Drawing and placing anchors (`place_theme_entries`)

A puzzle uses only `max_anchors` of the pool, drawn at random. `CrosswordGenerator.place_theme_entries` pins them into distinct slots of matching length, checking that letters agree at every shared intersection. It is best-effort: an anchor that cannot be placed consistently is skipped.

In [ ]:
gen = CrosswordGenerator(word_db)

def show_grid(grid):
    return print(pd.DataFrame([[c if c != '#' else '' for c in row] for row in grid]))

random.seed(0)
draw = random.sample(pool, cfg['max_anchors'])
working = gen.place_theme_entries(template, draw)
print('random draw from the pool:', draw)
print('placed:', working['filled_slots'])
show_grid(working['grid'])

## 5. Why draws are retried (and why the pool needs to be big)

Pinning several words in a 5x5 often leaves a crossing that no database word can complete, so a given draw may not fill. The generator simply **redraws** (`anchor_attempts` times per anchor count, then one fewer anchor). With a large pool there are plenty of alternative draws, so it nearly always lands two or three.

In [ ]:
def fill_rate(anchor_set, trials=10):
    ok = 0
    for s in range(trials):
        random.seed(s)
        w = gen.place_theme_entries(template, list(anchor_set))
        placed = w['filled_slots']
        if len(placed) == len(anchor_set) and gen.fill(template, seed_assignment=dict(placed)) is not None:
            ok += 1
    return f'{ok}/{trials}'

random.seed(1)
print('fill rate for a few random draws of 3:')
for _ in range(4):
    d = random.sample(pool, 3)
    print(f'  {d}: {fill_rate(d)}')
print('fill rate for a few random draws of 2:')
for _ in range(4):
    d = random.sample(pool, 2)
    print(f'  {d}: {fill_rate(d)}')

## 6. The payoff: a different puzzle every time

Because the pool is unranked and anchors are sampled, repeated generations for the *same* theme explore different parts of the space instead of settling on one "best" combination.

In [ ]:
grids, rows = set(), []
for s in range(6):
    random.seed(s)
    cw = generate_themed_crossword(
        template, theme_entries=pool, word_db_manager=word_db,
        max_anchors=cfg['max_anchors'], anchor_attempts=cfg['anchor_attempts'])
    grids.add(tuple(tuple(r) for r in cw['grid']))
    rows.append({'run': s,
                 'anchors placed': ', '.join(sorted(cw['seed_entries'].values())),
                 'all words': ', '.join(sorted(cw['filled_slots'].values()))})
print(f'distinct grids across {len(rows)} runs: {len(grids)}')
pd.DataFrame(rows).set_index('run')

## 7. The whole pipeline, as the backend runs it

This mirrors what the API route does for a themed request: one `ThemeManager` (shared theme embedding) scores the words and yields candidates, the `ThemeAnchorSelector` vets them into a pool, and `generate_themed_crossword` draws, places and fills. The similarity map also biases the remaining slots on-theme.

In [ ]:
def build_themed(theme, template_id='5x5_diagonal_cut', allow_llm_words=False, seed=0):
    tm = ThemeManager(theme, word_db)
    sims = tm.score_all_words()
    cands = tm.get_anchor_candidates(
        pool_size=cfg['candidate_pool'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])
    vetted = ThemeAnchorSelector().select_anchors(
        theme, cands, word_db, max_words=cfg['vetted_pool'], allow_llm_words=allow_llm_words,
        min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])
    random.seed(seed)
    cw = generate_themed_crossword(
        select_template(template_id=template_id), theme_entries=vetted, theme_similarities=sims,
        word_db_manager=word_db, max_anchors=cfg['max_anchors'], anchor_attempts=cfg['anchor_attempts'])
    return vetted, cw

for th in ['planets', 'sports', 'music']:
    vetted, cw = build_themed(th)
    print(f'{th:9} pool={len(vetted):2} words -> anchors placed: {sorted(cw["seed_entries"].values())}')